In [1]:
"""
QLoRA Fine-tuning Script for Llama 3 8B on Humor Generation
Optimized for Google Colab with limited resources
"""

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import wandb
from huggingface_hub import notebook_login

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Hyperparameters and configuration"""

    # Model settings
    MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

    # Dataset
    DATASET_PATH = "humor_training_data.jsonl"

    # QLoRA settings
    LORA_R = 16                    # LoRA attention dimension
    LORA_ALPHA = 32                # Alpha parameter for LoRA scaling
    LORA_DROPOUT = 0.05            # Dropout probability for LoRA layers

    # Target modules for LoRA (Llama 3 specific)
    TARGET_MODULES = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]

    # Training hyperparameters
    OUTPUT_DIR = "./llama3-humor-lora"
    NUM_EPOCHS = 3
    BATCH_SIZE = 4                 # Per device batch size
    GRADIENT_ACCUM_STEPS = 4       # Effective batch size = 4 * 4 = 16
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 512
    WARMUP_RATIO = 0.03
    LOGGING_STEPS = 10
    SAVE_STEPS = 100
    EVAL_STEPS = 50

    # Optimization
    OPTIM = "paged_adamw_8bit"     # Memory-efficient optimizer
    FP16 = False
    BF16 = True                     # Use bfloat16 if available

    # Data split
    TRAIN_SPLIT = 0.95

    # WandB (optional)
    USE_WANDB = False
    WANDB_PROJECT = "llama3-humor-generation"

    # Seed
    SEED = 42

config = Config()

In [3]:
def setup_environment():
    """Install required packages and login to Hugging Face"""

    print("🔧 Setting up environment...")

    # Check for GPU
    if torch.cuda.is_available():
        print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("⚠️  No GPU detected. Training will be very slow!")

    # Login to Hugging Face (needed for Llama 3)
    print("\n🔐 Please login to Hugging Face:")
    print("   You need access to Llama 3 model (request at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)")
    notebook_login()

    # Optional: WandB login
    if config.USE_WANDB:
        print("\n📊 Logging into Weights & Biases...")
        wandb.login()

    print("✓ Environment setup complete\n")

In [4]:
# ============================================================================
# DATA LOADING
# ============================================================================

def load_and_prepare_dataset(dataset_path):
    """Load dataset and split into train/test"""

    print(f"📂 Loading dataset from {dataset_path}...")

    # Load dataset
    dataset = load_dataset('json', data_files=dataset_path, split='train')

    print(f"  Total examples: {len(dataset)}")

    # Split into train/test
    dataset = dataset.train_test_split(
        test_size=1 - config.TRAIN_SPLIT,
        seed=config.SEED
    )

    print(f"  Train examples: {len(dataset['train'])}")
    print(f"  Test examples: {len(dataset['test'])}")

    # Show sample
    print("\n📋 Sample training example:")
    sample = dataset['train'][0]
    print(f"  User: {sample['messages'][0]['content'][:100]}...")
    print(f"  Assistant: {sample['messages'][1]['content'][:100]}...")

    return dataset

In [5]:
# ============================================================================
# MODEL LOADING
# ============================================================================

def load_model_and_tokenizer():
    """Load Llama 3 with 4-bit quantization for QLoRA"""

    print(f"\n🤖 Loading model: {config.MODEL_NAME}")

    # Quantization config for 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if config.BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,  # Nested quantization for more memory savings
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        config.MODEL_NAME,
        token=True,
        trust_remote_code=True
    )

    # Set padding token (Llama 3 doesn't have one by default)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "right"  # Required for training

    print("  ✓ Tokenizer loaded")

    # Load model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        token = True,
        trust_remote_code=True,
    )

    print("  ✓ Model loaded with 4-bit quantization")

    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)

    print("  ✓ Model prepared for k-bit training")

    return model, tokenizer

In [6]:
# ============================================================================
# LORA CONFIGURATION
# ============================================================================

def setup_lora(model):
    """Configure and apply LoRA to the model"""

    print("\n🔧 Setting up LoRA adapters...")

    lora_config = LoraConfig(
        r=config.LORA_R,
        lora_alpha=config.LORA_ALPHA,
        target_modules=config.TARGET_MODULES,
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)

    # Print trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    trainable_percent = 100 * trainable_params / all_params

    print(f"  ✓ LoRA adapters applied")
    print(f"  Trainable params: {trainable_params:,} ({trainable_percent:.2f}%)")
    print(f"  All params: {all_params:,}")

    return model


In [7]:
def format_chat_template(example, tokenizer):
    """
    Format examples using Llama 3's chat template.
    The dataset already has 'messages' field in the correct format.
    """
    # Apply chat template
    formatted_text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

In [8]:
from trl import SFTTrainer, SFTConfig

def train_model(model, tokenizer, dataset):
    print("\n🚀 Starting training...")

    # 1️⃣ Apply chat template once
    def apply_template(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}

    print("📝 Applying chat template to dataset...")
    train_dataset = dataset["train"].map(
        apply_template,
        remove_columns=dataset["train"].column_names,
        desc="Formatting train dataset",
    )

    eval_dataset = dataset["test"].map(
        apply_template,
        remove_columns=dataset["test"].column_names,
        desc="Formatting eval dataset",
    )

    print("✓ Datasets formatted")
    print(f"Sample:\n{train_dataset[0]['text'][:300]}")

    # 2️⃣ SFT config
    training_args = SFTConfig(
        output_dir=config.OUTPUT_DIR,
        num_train_epochs=config.NUM_EPOCHS,
        per_device_train_batch_size=config.BATCH_SIZE,
        per_device_eval_batch_size=config.BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUM_STEPS,
        gradient_checkpointing=True,
        optim=config.OPTIM,
        learning_rate=config.LEARNING_RATE,
        warmup_ratio=config.WARMUP_RATIO,
        lr_scheduler_type="cosine",
        logging_steps=config.LOGGING_STEPS,
        save_steps=config.SAVE_STEPS,
        eval_steps=config.EVAL_STEPS,
        eval_strategy="steps",
        save_strategy="steps",
        fp16=config.FP16,
        bf16=config.BF16,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="wandb" if config.USE_WANDB else "none",
        run_name=f"llama3-humor-{config.LORA_R}r" if config.USE_WANDB else None,
        seed=config.SEED,
        dataloader_num_workers=2,
        group_by_length=True,

        # SFT-specific
        max_length=config.MAX_SEQ_LENGTH,
        packing=False,
        dataset_text_field="text",  # ⭐ CRITICAL
    )

    # 3️⃣ Trainer — NO tokenizer / processing_class
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    print("\n" + "=" * 60)
    print("TRAINING STARTED")
    print("=" * 60)

    trainer.train()

    print("\n✓ TRAINING COMPLETE")

    trainer.save_model()
    tokenizer.save_pretrained(config.OUTPUT_DIR)

    return trainer

In [9]:
# ============================================================================
# INFERENCE TESTING
# ============================================================================

def test_model(model, tokenizer):
    """Test the fine-tuned model with sample inputs"""

    print("\n🧪 Testing fine-tuned model...")

    # Put model in evaluation mode
    model.eval()

    # Test cases
    test_cases = [
        ("telescope", "sandwich"),
        ("elephant", "programming"),
        ("coffee", "quantum"),
    ]

    for word1, word2 in test_cases:
        prompt = f"Generate a funny joke that naturally includes both of these words: '{word1}' and '{word2}'. The joke should be creative, humorous, and incorporate both words seamlessly."

        # Format with chat template
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract just the assistant's response
        if "assistant" in generated:
            joke = generated.split("assistant")[-1].strip()
        else:
            joke = generated[len(formatted_prompt):].strip()

        print(f"\n{'='*60}")
        print(f"Words: {word1}, {word2}")
        print(f"{'='*60}")
        print(f"Joke: {joke}")

    print("\n✓ Testing complete")

In [10]:
# ============================================================================
# MAIN PIPELINE
# ============================================================================

"""Run the complete fine-tuning pipeline"""

print("="*60)
print("LLAMA 3 HUMOR GENERATION - QLORA FINE-TUNING")
print("="*60)

# Setup
setup_environment()

LLAMA 3 HUMOR GENERATION - QLORA FINE-TUNING
🔧 Setting up environment...
✓ GPU detected: NVIDIA GeForce RTX 5090 Laptop GPU
  Memory: 25.65 GB

🔐 Please login to Hugging Face:
   You need access to Llama 3 model (request at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
✓ Environment setup complete



In [11]:
# Load data
dataset = load_and_prepare_dataset(config.DATASET_PATH)

📂 Loading dataset from humor_training_data.jsonl...
  Total examples: 10000
  Train examples: 9499
  Test examples: 501

📋 Sample training example:
  User: Write a joke that contains both 'exam' and 'muslims'. The joke should be funny and use both words in...
  Assistant: What's a Muslims favorite answer on a multiple choice exam? D) Allah the above...


In [12]:
from huggingface_hub import hf_hub_download

MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
path = hf_hub_download(repo_id=MODEL, filename="config.json", token=True)
print("OK:", path)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

OK: C:\Users\Anwender\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3-8B-Instruct\snapshots\8afb486c1db24fe5011ec46dfbe5b5dccdb575c2\config.json


In [13]:
# Load model



model, tokenizer = load_model_and_tokenizer()


🤖 Loading model: meta-llama/Meta-Llama-3-8B-Instruct


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

  ✓ Tokenizer loaded


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

  ✓ Model loaded with 4-bit quantization
  ✓ Model prepared for k-bit training


In [14]:
# Setup LoRA
model = setup_lora(model)


🔧 Setting up LoRA adapters...
  ✓ LoRA adapters applied
  Trainable params: 41,943,040 (0.92%)
  All params: 4,582,543,360


In [15]:
tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tokenize=False
)

"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWrite a joke that contains both 'exam' and 'muslims'. The joke should be funny and use both words in a creative way.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nWhat's a Muslims favorite answer on a multiple choice exam? D) Allah the above<|eot_id|>"

In [16]:
# # Train
trainer = train_model(model, tokenizer, dataset)


🚀 Starting training...
📝 Applying chat template to dataset...


Formatting train dataset:   0%|          | 0/9499 [00:00<?, ? examples/s]

Formatting eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Datasets formatted
Sample:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Write a joke that contains both 'exam' and 'muslims'. The joke should be funny and use both words in a creative way.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

What's a Muslims favorite answer on a multiple choice exam? D) Allah


...

Tokenizing eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]


TRAINING STARTED


Step,Training Loss,Validation Loss
50,1.469624,1.545114
100,1.274980,1.414967
150,1.204939,1.394393
200,1.218890,1.340591
250,1.184810,1.342480
300,1.199741,1.305099
350,1.175831,1.279439
400,1.176288,1.267104
450,1.170149,1.278577
500,1.137024,1.233459



✓ TRAINING COMPLETE


In [17]:
# # Test
test_model(model, tokenizer)

print("\n" + "="*60)
print("🎉 PIPELINE COMPLETE!")
print("="*60)
print(f"\nModel saved to: {config.OUTPUT_DIR}")
print("\nTo use your model:")
print("  1. Load with: model = AutoModelForCausalLM.from_pretrained(...)")
print("  2. Load LoRA: model = PeftModel.from_pretrained(model, 'path/to/lora')")
print("  3. Generate jokes!")


🧪 Testing fine-tuned model...

Words: telescope, sandwich
Joke: I just bought a telescope. I was hoping to see a sandwich.

Words: elephant, programming
Joke: How do you stop an elephant from eating your garden? Programming.

Words: coffee, quantum
Joke: I added quantum foam to my coffee this morning. It was a double shot of espresso.

✓ Testing complete

🎉 PIPELINE COMPLETE!

Model saved to: ./llama3-humor-lora

To use your model:
  1. Load with: model = AutoModelForCausalLM.from_pretrained(...)
  2. Load LoRA: model = PeftModel.from_pretrained(model, 'path/to/lora')
  3. Generate jokes!
